In [1]:
%run ./data.ipynb

loaded 920 rows, kept 740 after filtering
all the data checks look good


In [2]:
import hashlib, platform, time, traceback
from dataclasses import field
from functools import lru_cache

import torch
import opacus, sklearn
from opacus.accountants import RDPAccountant
from opacus.accountants.utils import get_noise_multiplier
from sklearn.metrics import roc_auc_score

torch.set_default_dtype(torch.float64)

In [3]:
# our experiment settings. we locked these in before the main run. experiments and results both use them
CONFIG = {
    "output_dir": "results",
    "data": {"source": "uci heart disease id 45, processed.{cleveland,hungarian,switzerland,va}.data",
             "split": "flamby fed-heart-disease (random_state 43, train 0.66)",
             "transform": "fixed bounds, chol zeros kept"},
    "train": {"rounds": 50, "local_steps": 5, "batch": 16, "pooled_batch": 64,
              "lr": 0.3,  # we picked this in experiments using training loss only
              "l2": 0.001, "clip": 1.0, "delta": 1e-05},
    "experiment": {"repetitions": 5, "epsilons": [1, 2, 4, 8], "lambda": 0.1,
                   "lambda_sweep": {"values": [0.0, 0.01, 0.1, 1.0], "eps": [None, 2, 8]},
                   "equal_weights": {"eps": [None, 2]},
                   "chol_sensitivity": {"eps": [None, 2]},
                   "flamby_parity": True,
                   "size_analysis": {"site": 0, "fractions": [0.25, 0.5, 1.0], "eps": [None, 2]}},
}
LAM = CONFIG["experiment"]["lambda"]

In [4]:
# privacy accounting with opacus
# each step we sample patients with probability q, clip each one's gradient to length C,
# add them up, add gaussian noise with std sigma * C, and divide by 16
# removing a patient just turns their slot into an empty record with zero gradient, so q and the divisor stay the same
@lru_cache(maxsize=None)
def calibrate_sigma(target_eps, delta, q, steps):
    sigma = get_noise_multiplier(target_epsilon=target_eps, target_delta=delta, sample_rate=q,
                                 steps=steps, accountant="rdp", epsilon_tolerance=0.01)
    while realized_epsilon(sigma, q, steps, delta) > target_eps:  # nudge it up if we're over budget
        sigma *= 1.001
    return float(sigma)


def realized_epsilon(sigma, q, steps, delta):
    acc = RDPAccountant()
    acc.history = [(sigma, q, steps)]
    return float(acc.get_epsilon(delta))


class SiteAccountant:
    # each hospital gets its own accountant per run, and we never reset it between rounds
    def __init__(self, sigma, q, delta):
        self.sigma, self.q, self.delta = sigma, q, delta
        self.acc = RDPAccountant()
        self.steps = 0

    def step(self):
        self.acc.step(noise_multiplier=self.sigma, sample_rate=self.q)
        self.steps += 1

    def epsilon(self):
        return float(self.acc.get_epsilon(self.delta)) if self.steps else 0.0

    def state_dict(self):
        return {"history": list(self.acc.history), "steps": self.steps, "sigma": self.sigma, "q": self.q, "delta": self.delta}

    @classmethod
    def from_state_dict(cls, s):
        a = cls(s["sigma"], s["q"], s["delta"])
        a.acc.history, a.steps = list(s["history"]), s["steps"]
        return a

In [5]:
# our model is logistic regression. theta holds the weights plus the bias at the end
@dataclass
class TrainConfig:
    rounds: int = 50
    local_steps: int = 5
    batch: int = 16
    pooled_batch: int = 64  # 4 times 16, so pooled does the same total work
    lr: float = 0.05
    l2: float = 1e-3  # on the weights, not the bias
    clip: float = 1.0
    delta: float = 1e-5

    @property
    def steps(self):
        return self.rounds * self.local_steps


def derive_seed(*parts):
    # every setting, run and hospital gets its own random stream so noise is never reused
    h = hashlib.sha256("|".join(map(str, parts)).encode()).digest()
    return int.from_bytes(h[:8], "little") & ((1 << 63) - 1)


def init_theta(d, seed):
    return torch.randn(d + 1, generator=torch.Generator().manual_seed(seed)) * 0.01


def logits(theta, X):
    return X @ theta[:-1] + theta[-1]


def per_sample_grads(theta, X, y):
    r = torch.sigmoid(logits(theta, X)) - y
    return torch.cat([r[:, None] * X, r[:, None]], dim=1)


def clip_rows(g, C):
    return g * torch.clamp(C / (g.norm(dim=1, keepdim=True) + 1e-12), max=1.0)


def noisy_grad(theta, X, y, present, q, B, clip, sigma, gen_sample, gen_noise):
    # one private step. present=False marks an empty slot. if nobody gets sampled, the step is just noise
    sel = (torch.rand(X.shape[0], generator=gen_sample) < q) & present
    g = per_sample_grads(theta, X[sel], y[sel])
    if clip is not None:
        g = clip_rows(g, clip)
    s = g.sum(dim=0)
    if sigma > 0:
        s = s + sigma * clip * torch.randn(s.shape, generator=gen_noise)
    return s / B, int(sel.sum())


def reg_grad(theta, l2):
    r = l2 * theta.clone()
    r[-1] = 0.0
    return r

In [6]:
class Client:
    # one simulated hospital with its own data, random streams, model and accountant
    def __init__(self, cid, name, X, y, cfg, mode, eps, run_key, batch=None):
        self.cid, self.name, self.cfg, self.mode = cid, name, cfg, mode
        self.X, self.y = torch.as_tensor(X), torch.as_tensor(y)
        self.N = self.X.shape[0]
        self.present = torch.ones(self.N, dtype=torch.bool)
        self.q = min(1.0, (cfg.batch if batch is None else batch) / self.N)
        self.B = self.q * self.N
        self.clip = cfg.clip if mode in ("dp", "clip_only") else None
        self.sigma, self.accountant = 0.0, None
        if mode == "dp":
            self.sigma = calibrate_sigma(eps, cfg.delta, self.q, cfg.steps)
            self.accountant = SiteAccountant(self.sigma, self.q, cfg.delta)
        self.gen_sample = torch.Generator().manual_seed(derive_seed(run_key, cid, "sample"))
        self.gen_noise = torch.Generator().manual_seed(derive_seed(run_key, cid, "noise"))
        self.theta, self.grad_evals, self.steps_done = None, 0, 0

    def run_local(self, theta_start, S, mu=None, lam=0.0):
        theta = theta_start.clone()
        for _ in range(S):
            g, n = noisy_grad(theta, self.X, self.y, self.present, self.q, self.B, self.clip, self.sigma,
                              self.gen_sample, self.gen_noise)
            upd = g + reg_grad(theta, self.cfg.l2)
            if mu is not None and lam > 0:
                upd = upd + lam * (theta - mu)  # the mr-mtl pull, added after the noise so it costs no extra privacy
            theta = theta - self.cfg.lr * upd
            self.grad_evals += n
            self.steps_done += 1
            if self.accountant is not None:
                self.accountant.step()
        self.theta = theta
        return theta


@dataclass
class FitResult:
    models: dict  # hospital id to the model we score there
    ledger: list = field(default_factory=list)
    elapsed: float = 0.0
    weights: dict = field(default_factory=dict)
    mu: object = None


def _ledger_row(c, cfg):
    return {"hospital_id": c.cid, "hospital_name": c.name, "n_train_slots": c.N, "sample_rate": c.q,
            "expected_batch": c.B, "clip_norm": c.clip, "noise_multiplier": c.sigma if c.accountant else None,
            "optimizer_steps": c.steps_done, "patient_grad_evals": c.grad_evals,
            "expected_patient_grad_evals": c.q * c.N * c.steps_done,
            "epsilon_realized": c.accountant.epsilon() if c.accountant else None,
            "accountant_steps": c.accountant.steps if c.accountant else None,
            "delta": cfg.delta if c.accountant else None}


def fit(sites, method, cfg, mode="nonprivate", eps=None, lam=0.0, weighting="size", rep=0, run_key=""):
    # method is local, fedavg, mrmtl or pooled. mode is nonprivate, clip_only or dp
    t0 = time.perf_counter()
    theta0 = init_theta(sites[0].X_train.shape[1], seed=1000 + rep)
    if method == "pooled":
        assert mode == "nonprivate"
        c = Client(-1, "pooled", np.concatenate([s.X_train for s in sites]), np.concatenate([s.y_train for s in sites]),
                   cfg, mode, eps, run_key, batch=cfg.pooled_batch)
        theta = c.run_local(theta0, cfg.steps)
        return FitResult({s.cid: theta for s in sites}, [_ledger_row(c, cfg)], time.perf_counter() - t0)

    clients = [Client(s.cid, s.name, s.X_train, s.y_train, cfg, mode, eps, run_key) for s in sites]
    tot = sum(c.N for c in clients)
    w = {c.cid: (c.N / tot if weighting == "size" else 1.0 / len(clients)) for c in clients}
    for c in clients:
        c.theta = theta0.clone()
    glob, mu = theta0.clone(), theta0.clone()
    for _ in range(cfg.rounds):
        if method == "local":
            for c in clients:
                c.run_local(c.theta, cfg.local_steps)
        elif method == "fedavg":
            for c in clients:
                c.run_local(glob, cfg.local_steps)  # every hospital starts the round from the shared model
            glob = sum(w[c.cid] * c.theta for c in clients)
        elif method == "mrmtl":
            mu_prev = mu.clone()
            for c in clients:
                c.run_local(c.theta, cfg.local_steps, mu=mu_prev, lam=lam)  # each hospital keeps its own model
            mu = sum(w[c.cid] * c.theta for c in clients)
        else:
            raise ValueError(method)
    models = {c.cid: glob for c in clients} if method == "fedavg" else {c.cid: c.theta for c in clients}
    return FitResult(models, [_ledger_row(c, cfg) for c in clients], time.perf_counter() - t0, w,
                     mu if method == "mrmtl" else None)

In [7]:
# how we score each hospital
def predict_proba(theta, X):
    with torch.no_grad():
        return torch.sigmoid(logits(theta, torch.as_tensor(X))).numpy()


def site_metrics(y, p, threshold=0.5):
    yhat, y = (p >= threshold).astype(int), y.astype(int)
    pos, neg = y == 1, y == 0
    sens = float((yhat[pos] == 1).mean()) if pos.any() else np.nan
    spec = float((yhat[neg] == 0).mean()) if neg.any() else np.nan
    both = pos.any() and neg.any()  # with only one class these scores don't exist, so we store nan instead of 0
    return {"accuracy": float((yhat == y).mean()), "balanced_accuracy": 0.5 * (sens + spec) if both else np.nan,
            "auroc": float(roc_auc_score(y, p)) if both else np.nan,
            "auroc_note": "" if both else "undefined: single class in test set",
            "sensitivity": sens, "specificity": spec,
            "n_test": int(len(y)), "n_test_positive": int(pos.sum()), "n_test_negative": int(neg.sum())}


def evaluate(models, sites):
    # each hospital's model gets scored on its own test patients
    rows, preds = [], []
    for s in sites:
        p = predict_proba(models[s.cid], s.X_test)
        rows.append({**site_metrics(s.y_test, p), "hospital_id": s.cid, "hospital_name": s.name, "n_train": len(s.y_train)})
        preds += [{"hospital_id": s.cid, "patient": rid, "y": int(yy), "p": float(pp)} for rid, yy, pp in zip(s.test_row_ids, s.y_test, p)]
    return rows, preds


def bootstrap_macro_ba(pred, n_boot=2000, seed=0, exclude=()):
    # we resample test patients within each hospital and class so both classes stay in.
    # every setting uses the same draws, so comparisons are paired
    rng = np.random.default_rng(seed)
    pred = pred[~pred.hospital_id.isin(exclude)]
    settings = list(pred.setting.unique())
    patients = pred[["hospital_id", "patient", "y"]].drop_duplicates().sort_values(["hospital_id", "patient"])
    reps = sorted(pred.repetition.unique())
    piv = pred.pivot_table(index="patient", columns=["setting", "repetition"], values="p").loc[patients.patient]
    P = np.stack([np.stack([piv[(s, r)].to_numpy() for r in reps]) for s in settings])  # setting by seed by patient
    y, hosp = patients.y.to_numpy(), patients.hospital_id.to_numpy()
    strata = [s for s in (np.flatnonzero((hosp == h) & (y == c)) for h in np.unique(hosp) for c in (0, 1)) if len(s)]

    def macro(idx):
        yy, hh, Pi = y[idx], hosp[idx], P[:, :, idx] >= 0.5
        vals = []
        for h in np.unique(hosp):
            pos, neg = (hh == h) & (yy == 1), (hh == h) & (yy == 0)
            if pos.any() and neg.any():
                vals.append(0.5 * (Pi[:, :, pos].mean(-1) + (~Pi[:, :, neg]).mean(-1)))
        return np.mean(vals, axis=0).mean(axis=1)

    point = macro(np.arange(len(y)))
    draws = np.array([macro(np.concatenate([rng.choice(s, size=len(s), replace=True) for s in strata])) for _ in range(n_boot)])
    out = pd.DataFrame({"setting": settings, "point": point,
                        "lo": np.percentile(draws, 2.5, axis=0), "hi": np.percentile(draws, 97.5, axis=0)})
    return {"summary": out, "draws": pd.DataFrame(draws, columns=settings)}

In [8]:
# we check the math here: gradients, fedavg, mr-mtl, clipping, noise, accounting and metrics
def tiny_sites(n=(12, 8), d=3, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for cid, k in enumerate(n):
        X, y = rng.uniform(-1, 1, (k, d)), (rng.uniform(size=k) < 0.5).astype(float)
        out.append(Site(cid, f"s{cid}", X, y, X[:4], y[:4], np.array([f"{cid}:{i}" for i in range(4)])))
    return out


def check_grads_match_autograd():
    torch.manual_seed(0)
    X, y, theta = torch.randn(7, 4), (torch.rand(7) < 0.5).double(), torch.randn(5)
    G = per_sample_grads(theta, X, y)
    lin = torch.nn.Linear(4, 1).double()
    with torch.no_grad():
        lin.weight.copy_(theta[:-1][None]); lin.bias.copy_(theta[-1:])
    for j in range(7):
        lin.zero_grad()
        torch.nn.functional.binary_cross_entropy_with_logits(lin(X[j:j + 1]).squeeze(1), y[j:j + 1]).backward()
        assert torch.allclose(G[j], torch.cat([lin.weight.grad.flatten(), lin.bias.grad]), atol=1e-12)


def check_fedavg_by_hand():
    s, cfg = tiny_sites(), TrainConfig(rounds=2, local_steps=3, batch=4, lr=0.1)
    res = fit(s, "fedavg", cfg, run_key="k")
    cl = [Client(x.cid, x.name, x.X_train, x.y_train, cfg, "nonprivate", None, "k") for x in s]
    glob = init_theta(3, 1000)
    for _ in range(2):
        th = [c.run_local(glob, 3) for c in cl]
        glob = 0.6 * th[0] + 0.4 * th[1]
    assert torch.allclose(res.models[0], glob) and torch.allclose(res.models[1], glob)


def check_mrmtl_by_hand():
    s, cfg = tiny_sites(), TrainConfig(rounds=2, local_steps=3, batch=4, lr=0.1)
    res = fit(s, "mrmtl", cfg, lam=0.5, run_key="k")
    cl = [Client(x.cid, x.name, x.X_train, x.y_train, cfg, "nonprivate", None, "k") for x in s]
    th0 = init_theta(3, 1000)
    th, mu = [th0.clone(), th0.clone()], th0.clone()
    for _ in range(2):
        th = [c.run_local(t, 3, mu=mu.clone(), lam=0.5) for c, t in zip(cl, th)]
        mu = 0.6 * th[0] + 0.4 * th[1]
    assert torch.allclose(res.models[0], th[0]) and torch.allclose(res.models[1], th[1])
    assert not torch.allclose(res.models[0], res.models[1])


def check_mrmtl_one_step():
    s = tiny_sites()[0]
    c = Client(0, "a", s.X_train, s.y_train, TrainConfig(batch=12, lr=0.2, l2=0.01), "nonprivate", None, "k")
    th, mu = torch.tensor([0.3, -0.2, 0.1, 0.05]), torch.tensor([0.0, 0.5, -0.5, 0.2])
    new = c.run_local(th, 1, mu=mu, lam=0.7)
    g = per_sample_grads(th, torch.as_tensor(s.X_train), torch.as_tensor(s.y_train)).sum(0) / 12
    reg = 0.01 * th.clone(); reg[-1] = 0
    assert torch.allclose(new, th - 0.2 * (g + reg + 0.7 * (th - mu)))


def check_lambda_zero_is_local():
    # this check shares random streams on purpose. our real runs never do
    s, cfg = tiny_sites(), TrainConfig(rounds=3, local_steps=2, batch=4)
    for mode, eps in [("nonprivate", None), ("dp", 2.0)]:
        a = fit(s, "mrmtl", cfg, mode=mode, eps=eps, lam=0.0, run_key="same")
        b = fit(s, "local", cfg, mode=mode, eps=eps, run_key="same")
        assert all(torch.allclose(a.models[c], b.models[c]) for c in a.models)


def check_one_client_fedavg_is_local():
    s, cfg = tiny_sites()[:1], TrainConfig(rounds=4, local_steps=3, batch=4)
    assert torch.allclose(fit(s, "fedavg", cfg, run_key="z").models[0], fit(s, "local", cfg, run_key="z").models[0])


def check_clipping():
    g = torch.randn(50, 6) * 10
    assert (clip_rows(g, 1.0).norm(dim=1) <= 1.0 + 1e-9).all()
    small = torch.full((1, 6), 0.01)
    assert torch.allclose(clip_rows(small, 1.0), small)


def check_empty_record():
    X, y, th = torch.randn(10, 3), torch.ones(10), torch.randn(4)
    present = torch.ones(10, dtype=torch.bool)
    absent = present.clone(); absent[3] = False
    g1, n1 = noisy_grad(th, X, y, present, 1.0, 10, 1.0, 0.0, torch.Generator().manual_seed(1), torch.Generator())
    g2, n2 = noisy_grad(th, X, y, absent, 1.0, 10, 1.0, 0.0, torch.Generator().manual_seed(1), torch.Generator())
    assert n1 - n2 == 1 and torch.allclose(g1 - g2, clip_rows(per_sample_grads(th, X[3:4], y[3:4]), 1.0)[0] / 10)
    ga, gb = torch.Generator().manual_seed(5), torch.Generator().manual_seed(5)
    noisy_grad(th, X, y, present, 0.3, 3, 1.0, 0.0, ga, torch.Generator())
    noisy_grad(th, X, y, absent, 0.3, 3, 1.0, 0.0, gb, torch.Generator())
    assert torch.equal(ga.get_state(), gb.get_state())  # the sampler stays in sync


def check_sampling_and_empty_batch():
    for s in sites:
        c = Client(s.cid, s.name, s.X_train, s.y_train, TrainConfig(), "dp", 2.0, "q")
        assert abs(c.q - 16 / len(s.y_train)) < 1e-12 and abs(c.B - 16) < 1e-9
    g, n = noisy_grad(torch.zeros(3), torch.randn(5, 2), torch.ones(5), torch.zeros(5, dtype=torch.bool), 0.5, 4.0, 2.0, 1.5,
                      torch.Generator(), torch.Generator().manual_seed(3))
    assert n == 0 and torch.allclose(g, 1.5 * 2.0 * torch.randn(3, generator=torch.Generator().manual_seed(3)) / 4.0)


def check_accounting():
    res = fit(sites, "fedavg", TrainConfig(rounds=10, local_steps=5), mode="dp", eps=2.0, run_key="acc")
    for l in res.ledger:
        assert l["accountant_steps"] == 50 == l["optimizer_steps"] and 1.95 < l["epsilon_realized"] <= 2.0
    a, eps = SiteAccountant(1.0, 0.1, 1e-5), []
    for _ in range(3):
        for _ in range(10):
            a.step()
        eps.append(a.epsilon())
    assert eps[0] < eps[1] < eps[2]
    b = SiteAccountant.from_state_dict(a.state_dict())
    assert abs(b.epsilon() - a.epsilon()) < 1e-12 and b.steps == 30
    sig = [calibrate_sigma(2.0, 1e-5, 16 / len(s.y_train), 250) for s in sites]
    assert len(set(np.round(sig, 6))) == 4 and sig[2] == max(sig)  # switzerland needs the most noise


def check_noise_scale():
    gn = torch.Generator().manual_seed(0)
    d = torch.stack([noisy_grad(torch.zeros(3), torch.zeros(4, 2), torch.zeros(4), torch.zeros(4, dtype=torch.bool),
                                0.5, 10.0, 0.5, 2.0, torch.Generator(), gn)[0] for _ in range(20000)])
    assert abs(d.std(0).mean().item() / (2.0 * 0.5 / 10.0) - 1) < 0.03 and d.mean(0).abs().max().item() < 0.005


def check_metrics():
    s = tiny_sites()
    models = {0: torch.tensor([5.0, 0, 0, 0]), 1: torch.tensor([-5.0, 0, 0, 0])}
    p = pd.DataFrame(evaluate(models, s)[1])
    for x in s:
        assert np.allclose(p[p.hospital_id == x.cid].p.to_numpy(), predict_proba(models[x.cid], x.X_test))
    m = site_metrics(np.ones(5), np.array([0.9, 0.8, 0.2, 0.7, 0.6]))
    assert np.isnan(m["auroc"]) and np.isnan(m["balanced_accuracy"])
    m2 = site_metrics(np.array([1, 1, 0]), np.array([0.9, 0.4, 0.2]))
    assert abs(m2["balanced_accuracy"] - 0.75) < 1e-12 and abs(m2["auroc"] - 1.0) < 1e-12


CHECKS = [check_grads_match_autograd, check_fedavg_by_hand, check_mrmtl_by_hand, check_mrmtl_one_step,
          check_lambda_zero_is_local, check_one_client_fedavg_is_local, check_clipping, check_empty_record,
          check_sampling_and_empty_batch, check_accounting, check_noise_scale, check_metrics]
for f in CHECKS:
    f()
print(f"all {len(CHECKS)} method checks look good")

all 12 method checks look good
